In [ ]:
import sounddevice as sd
import numpy as np
import torch

In [ ]:
sd._terminate()
sd._initialize()
print(sd.query_devices())

In [ ]:
# Get samplerate of devices:

for idx in [9, 8]:
    info = sd.query_devices(idx)
    print(f"Device {idx}: {info['name']}")
    print(f"  Default samplerate: {info['default_samplerate']}")

In [ ]:
# Edit processing to use ML model
def process_audio(indata, outdata, frames, time, status, model):
    data = torch.tensor(indata)
    if status:
        print(status)
    outdata[:] = model(data)


def no_processing(indata, outdata, frames, time, status):
    if status:
        print(status)
    outdata[:] = indata

def stream_audio(input_device = 9, output_device = 8):
    """Use sounddevice.query_devices to select input_device and output_device.
    For windows, choose WASAPI devices. For Mac, try PortAudio devices."""
    wasapi_exclusive = sd.WasapiSettings(exclusive=True)

    with sd.Stream(
        device=(input_device, output_device),
        channels=1,
        samplerate=48000,
        blocksize=512,          # Blocksize should match seq_len used for training
        latency="low",
        dtype="float32",
        extra_settings=wasapi_exclusive,
        callback=no_processing
    ):
        print("Running. Press Enter to stop.")
        input()

In [ ]:
stream_audio()